# **JAX Benchmark**

## Setup & GPU Info

In [1]:
# ============================================================
# JAX BENCHMARK - Cell 1: Setup & GPU Info
# ============================================================
import jax
import jax.numpy as jnp
from jax import random, jit, grad, vmap, value_and_grad
import flax.linen as nn
import optax
from flax.training import train_state
import numpy as np
import time
import json
import gc
import subprocess
import datetime
from functools import partial

print("=" * 70)
print("JAX BENCHMARK SUITE (ROCm)")
print("=" * 70)

print(f"\nJAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")

for i, dev in enumerate(jax.devices()):
    print(f"  Device {i}: {dev}")

try:
    import flax
    print(f"Flax version: {flax.__version__}")
except:
    print("Flax version: N/A")

try:
    print(f"Optax version: {optax.__version__}")
except:
    pass

# ROCm info
try:
    result = subprocess.run(['rocm-smi', '--showid', '--showtemp', '--showuse'],
                          capture_output=True, text=True, timeout=10)
    print(f"\nROCm SMI Output:\n{result.stdout}")
except:
    print("\nCould not run rocm-smi")

# Results storage
results = {
    'framework': 'JAX',
    'version': jax.__version__,
    'device': str(jax.devices()[0]),
    'benchmarks': {}
}

# Utility functions
def benchmark_fn(fn, num_runs=100, warmup_runs=20, label=""):
    """Benchmark a JAX function with proper synchronization"""
    # Warmup
    for _ in range(warmup_runs):
        result = fn()
        if isinstance(result, jnp.ndarray):
            result.block_until_ready()
        elif isinstance(result, (tuple, list)):
            for r in result:
                if isinstance(r, jnp.ndarray):
                    r.block_until_ready()

    times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        result = fn()
        # Ensure GPU synchronization
        if isinstance(result, jnp.ndarray):
            result.block_until_ready()
        elif isinstance(result, (tuple, list)):
            for r in result:
                if isinstance(r, jnp.ndarray):
                    r.block_until_ready()
        elif isinstance(result, dict):
            for r in result.values():
                if isinstance(r, jnp.ndarray):
                    r.block_until_ready()
        end = time.perf_counter()
        times.append(end - start)

    times = np.array(times)
    r = {
        'mean_ms': float(np.mean(times) * 1000),
        'std_ms': float(np.std(times) * 1000),
        'min_ms': float(np.min(times) * 1000),
        'max_ms': float(np.max(times) * 1000),
        'median_ms': float(np.median(times) * 1000),
        'p95_ms': float(np.percentile(times, 95) * 1000),
        'p99_ms': float(np.percentile(times, 99) * 1000),
        'num_runs': num_runs
    }
    if label:
        print(f"  {label}: {r['mean_ms']:.3f} ± {r['std_ms']:.3f} ms "
              f"(min={r['min_ms']:.3f}, median={r['median_ms']:.3f})")
    return r

def warmup_gpu():
    """Warmup GPU"""
    key = random.PRNGKey(0)
    for _ in range(10):
        a = random.normal(key, (256, 256))
        b = random.normal(key, (256, 256))
        c = jnp.dot(a, b)
        c.block_until_ready()
    del a, b, c

print("\n✅ Setup complete!")

JAX BENCHMARK SUITE (ROCm)

JAX version: 0.7.1
JAX backend: gpu
JAX devices: [RocmDevice(id=0), RocmDevice(id=1)]
Device count: 2
  Device 0: rocm:0
  Device 1: rocm:1
Flax version: 0.12.4
Optax version: 0.2.7

ROCm SMI Output:


============================ ROCm System Management Interface ============================
=========================================== ID ===========================================
GPU[0]		: Device Name: 		AMD Radeon RX 6800S
GPU[0]		: Device ID: 		0x73ef
GPU[0]		: Device Rev: 		0xc0
GPU[0]		: Subsystem ID: 	0x1dcc
GPU[0]		: GUID: 		45168
GPU[1]		: Device Name: 		AMD Radeon 680M
GPU[1]		: Device ID: 		0x1681
GPU[1]		: Device Rev: 		0xc7
GPU[1]		: Subsystem ID: 	0x1dcc
GPU[1]		: GUID: 		56463
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 51.0
GPU[0]		: Temperature (Sensor junction) (C): 51.0
GPU[0]		: Temperature (Sensor memory) (C): 48.0
GPU[1]		: Temperature (Sensor edge) (

## Matrix Operations

In [2]:
# ============================================================
# JAX BENCHMARK - Cell 2: Matrix Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 1: MATRIX OPERATIONS")
print("=" * 70)

results['benchmarks']['matmul'] = {}

key = random.PRNGKey(42)
sizes = [256, 512, 1024, 2048, 4096, 8192]
dtypes_config = {
    'float32': jnp.float32,
    'float16': jnp.float16,
    'bfloat16': jnp.bfloat16,
}

for dtype_name, dtype in dtypes_config.items():
    print(f"\n--- Matrix Multiplication ({dtype_name}) ---")
    results['benchmarks']['matmul'][dtype_name] = {}

    for size in sizes:
        try:
            warmup_gpu()
            key1, key2, key = random.split(key, 3)
            a = random.normal(key1, (size, size), dtype=dtype)
            b = random.normal(key2, (size, size), dtype=dtype)

            @jit
            def matmul_fn(a, b):
                return jnp.dot(a, b)

            # Pre-compile
            _ = matmul_fn(a, b).block_until_ready()

            def run_matmul():
                return matmul_fn(a, b)

            num_runs = 200 if size <= 1024 else (100 if size <= 4096 else 50)
            r = benchmark_fn(run_matmul, num_runs=num_runs, warmup_runs=20,
                           label=f"  Size {size}x{size}")

            flops = 2 * size * size * size
            tflops = flops / (r['mean_ms'] / 1000) / 1e12
            r['tflops'] = float(tflops)
            print(f"    → {tflops:.2f} TFLOPS")

            results['benchmarks']['matmul'][dtype_name][str(size)] = r

            del a, b
        except Exception as e:
            print(f"  Size {size}x{size}: SKIPPED ({e})")
            results['benchmarks']['matmul'][dtype_name][str(size)] = {'error': str(e)}

# Batched matmul
print(f"\n--- Batched Matrix Multiplication (float32) ---")
results['benchmarks']['batched_matmul'] = {}
batch_configs = [(32, 512, 512), (64, 256, 256), (128, 128, 128), (16, 1024, 1024)]

for batch, m, n in batch_configs:
    try:
        warmup_gpu()
        key1, key2, key = random.split(key, 3)
        a = random.normal(key1, (batch, m, n))
        b = random.normal(key2, (batch, n, m))

        @jit
        def batched_matmul_fn(a, b):
            return jnp.matmul(a, b)

        _ = batched_matmul_fn(a, b).block_until_ready()

        def run_batched():
            return batched_matmul_fn(a, b)

        r = benchmark_fn(run_batched, num_runs=100, warmup_runs=20,
                       label=f"  Batch={batch}, Size={m}x{n}")

        flops = 2 * batch * m * n * m
        tflops = flops / (r['mean_ms'] / 1000) / 1e12
        r['tflops'] = float(tflops)
        print(f"    → {tflops:.2f} TFLOPS")

        results['benchmarks']['batched_matmul'][f"b{batch}_{m}x{n}"] = r

        del a, b
    except Exception as e:
        print(f"  Config ({batch},{m},{n}): SKIPPED ({e})")

gc.collect()
print("\n✅ Matrix operations benchmark complete!")

BENCHMARK 1: MATRIX OPERATIONS

--- Matrix Multiplication (float32) ---
    Size 256x256: 0.086 ± 0.020 ms (min=0.072, median=0.083)
    → 0.39 TFLOPS
    Size 512x512: 0.116 ± 0.005 ms (min=0.105, median=0.114)
    → 2.32 TFLOPS
    Size 1024x1024: 0.389 ± 0.714 ms (min=0.311, median=0.336)
    → 5.53 TFLOPS
    Size 2048x2048: 2.641 ± 2.207 ms (min=2.004, median=2.143)
    → 6.50 TFLOPS
    Size 4096x4096: 22.161 ± 5.033 ms (min=16.618, median=18.568)
    → 6.20 TFLOPS
    Size 8192x8192: 187.068 ± 4.685 ms (min=179.326, median=189.977)
    → 5.88 TFLOPS

--- Matrix Multiplication (float16) ---
    Size 256x256: 0.113 ± 0.529 ms (min=0.065, median=0.070)
    → 0.30 TFLOPS
    Size 512x512: 0.132 ± 0.444 ms (min=0.082, median=0.099)
    → 2.04 TFLOPS
    Size 1024x1024: 0.296 ± 0.757 ms (min=0.197, median=0.216)
    → 7.25 TFLOPS
    Size 2048x2048: 1.435 ± 1.723 ms (min=1.065, median=1.123)
    → 11.97 TFLOPS


E0217 13:38:33.425686  149579 buffer_comparator.cc:150] Difference at 98078: 10.5078, expected 9.35156
E0217 13:38:33.425715  149579 buffer_comparator.cc:150] Difference at 98229: 10.5625, expected 9.40625
E0217 13:38:33.426183  149579 buffer_comparator.cc:150] Difference at 197750: 10.4844, expected 9.32812
E0217 13:38:33.426194  149579 buffer_comparator.cc:150] Difference at 198094: 10.2422, expected 9.11719
E0217 13:38:33.426247  149579 buffer_comparator.cc:150] Difference at 205127: 10.4375, expected 9.28906
E0217 13:38:33.426256  149579 buffer_comparator.cc:150] Difference at 205848: 10.4688, expected 9.29688
E0217 13:38:33.426264  149579 buffer_comparator.cc:150] Difference at 206167: 10.4141, expected 9.25
E0217 13:38:33.426274  149579 buffer_comparator.cc:150] Difference at 206992: 10.5625, expected 9.40625
E0217 13:38:33.426286  149579 buffer_comparator.cc:150] Difference at 208036: 10.3984, expected 9.24219
E0217 13:38:33.426338  149579 buffer_comparator.cc:150] Difference at

    Size 4096x4096: 11.836 ± 4.235 ms (min=8.699, median=9.687)
    → 11.61 TFLOPS


E0217 13:39:31.987863  149579 buffer_comparator.cc:150] Difference at 0: 20.8125, expected 16.375
E0217 13:39:31.987890  149579 buffer_comparator.cc:150] Difference at 1: 20.7188, expected 16.2969
E0217 13:39:31.987892  149579 buffer_comparator.cc:150] Difference at 2: 20.8594, expected 16.375
E0217 13:39:31.987894  149579 buffer_comparator.cc:150] Difference at 3: 20.75, expected 16.375
E0217 13:39:31.987897  149579 buffer_comparator.cc:150] Difference at 4: 20.5, expected 16.2656
E0217 13:39:31.987899  149579 buffer_comparator.cc:150] Difference at 5: 20.75, expected 16.375
E0217 13:39:31.987900  149579 buffer_comparator.cc:150] Difference at 6: 20.7969, expected 16.3125
E0217 13:39:31.987902  149579 buffer_comparator.cc:150] Difference at 7: 20.8281, expected 16.3438
E0217 13:39:31.987904  149579 buffer_comparator.cc:150] Difference at 8: 20.875, expected 16.4062
E0217 13:39:31.987906  149579 buffer_comparator.cc:150] Difference at 9: 20.625, expected 16.2812
2026-02-17 13:39:32.004

    Size 8192x8192: 95.779 ± 3.053 ms (min=86.423, median=96.754)
    → 11.48 TFLOPS

--- Matrix Multiplication (bfloat16) ---
    Size 256x256: 0.126 ± 0.448 ms (min=0.072, median=0.095)
    → 0.27 TFLOPS
    Size 512x512: 0.172 ± 0.364 ms (min=0.136, median=0.144)
    → 1.56 TFLOPS
    Size 1024x1024: 0.735 ± 1.229 ms (min=0.537, median=0.578)
    → 2.92 TFLOPS
    Size 2048x2048: 5.229 ± 3.058 ms (min=3.955, median=4.230)
    → 3.29 TFLOPS
    Size 4096x4096: 41.013 ± 3.964 ms (min=32.744, median=42.911)
    → 3.35 TFLOPS
    Size 8192x8192: 326.700 ± 5.173 ms (min=320.479, median=323.013)
    → 3.37 TFLOPS

--- Batched Matrix Multiplication (float32) ---
    Batch=32, Size=512x512: 1.498 ± 1.724 ms (min=1.104, median=1.200)
    → 5.73 TFLOPS
    Batch=64, Size=256x256: 0.475 ± 1.011 ms (min=0.353, median=0.368)
    → 4.53 TFLOPS
    Batch=128, Size=128x128: 0.251 ± 1.004 ms (min=0.132, median=0.147)
    → 2.14 TFLOPS
    Batch=16, Size=1024x1024: 5.522 ± 3.149 ms (min=3.907, median

## Element-wise & Reduction Operations

In [3]:
# ============================================================
# JAX BENCHMARK - Cell 3: Element-wise & Reduction Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 2: ELEMENT-WISE & REDUCTION OPERATIONS")
print("=" * 70)

results['benchmarks']['elementwise'] = {}
results['benchmarks']['reductions'] = {}

key = random.PRNGKey(123)
tensor_sizes = [
    (1_000_000,),
    (10_000_000,),
    (50_000_000,),
    (100_000_000,),
]

print("\n--- Element-wise Operations ---")
for shape in tensor_sizes:
    numel = shape[0]
    label = f"{numel/1e6:.0f}M"
    results['benchmarks']['elementwise'][label] = {}

    warmup_gpu()
    key1, key2, key = random.split(key, 3)
    a = random.normal(key1, shape)
    b = random.normal(key2, shape)

    ops = {
        'add': jit(lambda a, b: jnp.add(a, b)),
        'mul': jit(lambda a, b: jnp.multiply(a, b)),
        'exp': jit(lambda a: jnp.exp(a)),
        'sin': jit(lambda a: jnp.sin(a)),
        'sigmoid': jit(lambda a: jax.nn.sigmoid(a)),
        'tanh': jit(lambda a: jnp.tanh(a)),
        'relu': jit(lambda a: jax.nn.relu(a)),
        'gelu': jit(lambda a: jax.nn.gelu(a)),
        'silu': jit(lambda a: jax.nn.silu(a)),
    }

    print(f"\n  Tensor size: {label} elements")
    for op_name, op_fn in ops.items():
        try:
            if op_name in ['add', 'mul']:
                _ = op_fn(a, b).block_until_ready()
                run_fn = lambda: op_fn(a, b)
            else:
                _ = op_fn(a).block_until_ready()
                run_fn = lambda: op_fn(a)

            r = benchmark_fn(run_fn, num_runs=100, warmup_runs=20,
                           label=f"    {op_name}")
            bytes_moved = numel * 4 * 2
            if op_name in ['add', 'mul']:
                bytes_moved = numel * 4 * 3
            bandwidth_gbs = bytes_moved / (r['mean_ms'] / 1000) / 1e9
            r['bandwidth_gbs'] = float(bandwidth_gbs)
            results['benchmarks']['elementwise'][label][op_name] = r
        except Exception as e:
            print(f"    {op_name}: ERROR - {e}")

    del a, b

# Reductions
print(f"\n--- Reduction Operations ---")
for shape in tensor_sizes:
    numel = shape[0]
    label = f"{numel/1e6:.0f}M"
    results['benchmarks']['reductions'][label] = {}

    warmup_gpu()
    key, subkey = random.split(key)
    a = random.normal(subkey, shape)

    ops = {
        'sum': jit(lambda a: jnp.sum(a)),
        'mean': jit(lambda a: jnp.mean(a)),
        'max': jit(lambda a: jnp.max(a)),
        'min': jit(lambda a: jnp.min(a)),
        'std': jit(lambda a: jnp.std(a)),
        'norm': jit(lambda a: jnp.linalg.norm(a)),
        'argmax': jit(lambda a: jnp.argmax(a)),
        'sort': jit(lambda a: jnp.sort(a)),
    }

    print(f"\n  Tensor size: {label} elements")
    for op_name, op_fn in ops.items():
        try:
            _ = op_fn(a).block_until_ready()
            run_fn = lambda: op_fn(a)

            num_runs = 50 if op_name == 'sort' else 100
            r = benchmark_fn(run_fn, num_runs=num_runs, warmup_runs=10,
                           label=f"    {op_name}")
            results['benchmarks']['reductions'][label][op_name] = r
        except Exception as e:
            print(f"    {op_name}: ERROR - {e}")

    del a

gc.collect()
print("\n✅ Element-wise & reduction benchmark complete!")

BENCHMARK 2: ELEMENT-WISE & REDUCTION OPERATIONS

--- Element-wise Operations ---

  Tensor size: 1M elements
      add: 0.141 ± 0.647 ms (min=0.052, median=0.074)
      mul: 0.149 ± 0.716 ms (min=0.057, median=0.079)
      exp: 0.154 ± 0.726 ms (min=0.051, median=0.077)
      sin: 0.055 ± 0.010 ms (min=0.048, median=0.049)
      sigmoid: 0.053 ± 0.009 ms (min=0.047, median=0.048)
      tanh: 0.145 ± 0.758 ms (min=0.050, median=0.070)
      relu: 0.141 ± 0.763 ms (min=0.049, median=0.068)
      gelu: 0.129 ± 0.620 ms (min=0.048, median=0.070)
      silu: 0.144 ± 0.751 ms (min=0.049, median=0.070)

  Tensor size: 10M elements
      add: 0.701 ± 1.007 ms (min=0.584, median=0.597)
      mul: 0.803 ± 1.417 ms (min=0.575, median=0.598)
      exp: 0.637 ± 1.415 ms (min=0.409, median=0.428)
      sin: 0.531 ± 1.005 ms (min=0.401, median=0.428)
      sigmoid: 0.535 ± 1.008 ms (min=0.388, median=0.429)
      tanh: 0.535 ± 1.009 ms (min=0.411, median=0.431)
      relu: 0.549 ± 1.013 ms (min=0.39

## Convolution Operations

In [4]:
# ============================================================
# JAX BENCHMARK - Cell 4: Convolution Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 3: CONVOLUTION OPERATIONS")
print("=" * 70)

results['benchmarks']['conv2d'] = {}

key = random.PRNGKey(456)

conv_configs = [
    (32, 3, 64, 224, 224, 7, 2, 3, "ResNet-stem"),
    (32, 64, 64, 56, 56, 3, 1, 1, "ResNet-block1"),
    (32, 128, 128, 28, 28, 3, 1, 1, "ResNet-block2"),
    (32, 256, 256, 14, 14, 3, 1, 1, "ResNet-block3"),
    (32, 512, 512, 7, 7, 3, 1, 1, "ResNet-block4"),
    (64, 3, 32, 32, 32, 3, 1, 1, "CIFAR-style"),
    (16, 3, 64, 224, 224, 3, 1, 1, "VGG-style"),
    (1, 64, 64, 512, 512, 3, 1, 1, "HighRes-single"),
]

for batch, in_ch, out_ch, H, W, kernel_size, stride, padding, label in conv_configs:
    try:
        warmup_gpu()
        key, subkey = random.split(key)

        # JAX uses NHWC by default
        x = random.normal(subkey, (batch, H, W, in_ch))

        conv = nn.Conv(features=out_ch, kernel_size=(kernel_size, kernel_size),
                      strides=(stride, stride), padding=((padding, padding), (padding, padding)),
                      use_bias=False)

        key, subkey = random.split(key)
        params = conv.init(subkey, x)

        @jit
        def conv_forward(params, x):
            return conv.apply(params, x)

        _ = conv_forward(params, x).block_until_ready()

        print(f"\n  {label}: input=({batch},{in_ch},{H},{W}), "
              f"kernel={kernel_size}x{kernel_size}, out_ch={out_ch}")

        def run_conv_fwd():
            return conv_forward(params, x)

        r_fwd = benchmark_fn(run_conv_fwd, num_runs=100, warmup_runs=20,
                            label=f"    Forward")

        # Forward + Backward
        @jit
        def conv_fwd_bwd(params, x):
            def loss_fn(params):
                out = conv.apply(params, x)
                return jnp.sum(out)
            loss, grads = value_and_grad(loss_fn)(params)
            return loss, grads

        _ = conv_fwd_bwd(params, x)
        for v in jax.tree_util.tree_leaves(_):
            if isinstance(v, jnp.ndarray):
                v.block_until_ready()

        def run_conv_fwd_bwd():
            return conv_fwd_bwd(params, x)

        r_fwd_bwd = benchmark_fn(run_conv_fwd_bwd, num_runs=50, warmup_runs=10,
                                label=f"    Fwd+Bwd")

        results['benchmarks']['conv2d'][label] = {
            'forward': r_fwd,
            'fwd_bwd': r_fwd_bwd,
            'config': {
                'batch': batch, 'in_ch': in_ch, 'out_ch': out_ch,
                'H': H, 'W': W, 'kernel': kernel_size, 'stride': stride
            }
        }

        del x, params
    except Exception as e:
        print(f"  {label}: SKIPPED ({e})")
        results['benchmarks']['conv2d'][label] = {'error': str(e)}

gc.collect()
print("\n✅ Convolution benchmark complete!")

BENCHMARK 3: CONVOLUTION OPERATIONS

  ResNet-stem: input=(32,3,224,224), kernel=7x7, out_ch=64
      Forward: 2.878 ± 2.403 ms (min=2.208, median=2.259)
      Fwd+Bwd: 46.356 ± 2.726 ms (min=36.987, median=47.126)

  ResNet-block1: input=(32,64,56,56), kernel=3x3, out_ch=64
      Forward: 1.321 ± 1.730 ms (min=0.993, median=1.010)
      Fwd+Bwd: 3.822 ± 2.756 ms (min=2.876, median=3.013)

  ResNet-block2: input=(32,128,28,28), kernel=3x3, out_ch=128
      Forward: 0.959 ± 1.456 ms (min=0.711, median=0.725)
      Fwd+Bwd: 2.039 ± 1.993 ms (min=1.585, median=1.618)

  ResNet-block3: input=(32,256,14,14), kernel=3x3, out_ch=256
      Forward: 0.999 ± 1.421 ms (min=0.753, median=0.794)
      Fwd+Bwd: 1.798 ± 1.987 ms (min=1.345, median=1.381)

  ResNet-block4: input=(32,512,7,7), kernel=3x3, out_ch=512
      Forward: 3.159 ± 2.444 ms (min=2.259, median=2.386)
      Fwd+Bwd: 4.213 ± 2.804 ms (min=3.211, median=3.368)

  CIFAR-style: input=(64,3,32,32), kernel=3x3, out_ch=32
      Forward: 

## Common Layer Operations

In [5]:
# ============================================================
# JAX BENCHMARK - Cell 5: Common Layer Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 4: COMMON LAYER OPERATIONS")
print("=" * 70)

results['benchmarks']['layers'] = {}
key = random.PRNGKey(789)
warmup_gpu()

# --- Dense (Linear) Layer ---
print("\n--- Dense (Linear) Layers ---")
linear_configs = [
    (128, 768, 768, "BERT-hidden"),
    (128, 768, 3072, "BERT-FFN-up"),
    (128, 3072, 768, "BERT-FFN-down"),
    (32, 1024, 4096, "Large-FFN-up"),
    (32, 4096, 1024, "Large-FFN-down"),
    (1, 4096, 4096, "LLM-single"),
]

results['benchmarks']['layers']['linear'] = {}
for batch, in_feat, out_feat, label in linear_configs:
    try:
        warmup_gpu()
        key, k1, k2 = random.split(key, 3)
        x = random.normal(k1, (batch, in_feat))
        layer = nn.Dense(features=out_feat, use_bias=True)
        params = layer.init(k2, x)

        @jit
        def linear_fwd(params, x):
            return layer.apply(params, x)

        @jit
        def linear_fwd_bwd(params, x):
            def loss_fn(params):
                return jnp.sum(layer.apply(params, x))
            return value_and_grad(loss_fn)(params)

        _ = linear_fwd(params, x).block_until_ready()
        _ = linear_fwd_bwd(params, x)

        r_fwd = benchmark_fn(lambda: linear_fwd(params, x),
                            num_runs=200, warmup_runs=30, label=f"  {label} fwd")
        r_bwd = benchmark_fn(lambda: linear_fwd_bwd(params, x),
                            num_runs=100, warmup_runs=20, label=f"  {label} fwd+bwd")

        results['benchmarks']['layers']['linear'][label] = {
            'forward': r_fwd, 'fwd_bwd': r_bwd
        }
        del x, params
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- BatchNorm ---
print("\n--- BatchNorm ---")
results['benchmarks']['layers']['batchnorm'] = {}
bn_configs = [
    (32, 64, 56, 56, "BN-early"),
    (32, 256, 14, 14, "BN-mid"),
    (32, 512, 7, 7, "BN-late"),
]

for batch, ch, H, W, label in bn_configs:
    try:
        warmup_gpu()
        key, k1, k2 = random.split(key, 3)
        x = random.normal(k1, (batch, H, W, ch))  # NHWC
        bn = nn.BatchNorm(use_running_average=False)
        variables = bn.init(k2, x)

        @jit
        def bn_fwd(variables, x):
            return bn.apply(variables, x, mutable=['batch_stats'])

        _ = bn_fwd(variables, x)

        r = benchmark_fn(lambda: bn_fwd(variables, x),
                        num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['batchnorm'][label] = r
        del x, variables
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- LayerNorm ---
print("\n--- LayerNorm ---")
results['benchmarks']['layers']['layernorm'] = {}
ln_configs = [
    ((128, 128, 768), "BERT-LN"),
    ((32, 256, 1024), "Large-LN"),
    ((1, 2048, 4096), "LLM-LN"),
]

for shape, label in ln_configs:
    try:
        warmup_gpu()
        key, k1, k2 = random.split(key, 3)
        x = random.normal(k1, shape)
        ln = nn.LayerNorm()
        params = ln.init(k2, x)

        @jit
        def ln_fwd(params, x):
            return ln.apply(params, x)

        _ = ln_fwd(params, x).block_until_ready()

        r = benchmark_fn(lambda: ln_fwd(params, x),
                        num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['layernorm'][label] = r
        del x, params
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- Multi-Head Attention ---
print("\n--- Multi-Head Attention ---")
results['benchmarks']['layers']['attention'] = {}
attn_configs = [
    (32, 128, 768, 12, "BERT-attn"),
    (16, 256, 768, 12, "BERT-long-attn"),
    (8, 512, 1024, 16, "Large-attn"),
    (1, 1024, 1024, 16, "LLM-attn"),
    (1, 2048, 768, 12, "Long-seq-attn"),
]

for batch, seq_len, embed, heads, label in attn_configs:
    try:
        warmup_gpu()
        key, k1, k2 = random.split(key, 3)

        mha = nn.MultiHeadDotProductAttention(num_heads=heads, qkv_features=embed)
        x = random.normal(k1, (batch, seq_len, embed))
        params = mha.init(k2, x, x)

        @jit
        def attn_fwd(params, x):
            return mha.apply(params, x, x)

        @jit
        def attn_fwd_bwd(params, x):
            def loss_fn(params):
                return jnp.sum(mha.apply(params, x, x))
            return value_and_grad(loss_fn)(params)

        _ = attn_fwd(params, x).block_until_ready()
        _ = attn_fwd_bwd(params, x)

        r_fwd = benchmark_fn(lambda: attn_fwd(params, x),
                            num_runs=50, warmup_runs=10, label=f"  {label} fwd")
        r_bwd = benchmark_fn(lambda: attn_fwd_bwd(params, x),
                            num_runs=30, warmup_runs=10, label=f"  {label} fwd+bwd")

        results['benchmarks']['layers']['attention'][label] = {
            'forward': r_fwd, 'fwd_bwd': r_bwd
        }
        del x, params
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- Softmax ---
print("\n--- Softmax ---")
results['benchmarks']['layers']['softmax'] = {}
for size in [(128, 128, 768), (32, 256, 1024), (8, 512, 4096)]:
    try:
        warmup_gpu()
        key, subkey = random.split(key)
        x = random.normal(subkey, size)
        label_str = f"{'x'.join(map(str, size))}"

        @jit
        def softmax_fn(x):
            return jax.nn.softmax(x, axis=-1)

        _ = softmax_fn(x).block_until_ready()

        r = benchmark_fn(lambda: softmax_fn(x),
                        num_runs=200, warmup_runs=30, label=f"  {label_str}")
        results['benchmarks']['layers']['softmax'][label_str] = r
        del x
    except Exception as e:
        print(f"  {size}: ERROR - {e}")

gc.collect()
print("\n✅ Layer operations benchmark complete!")

BENCHMARK 4: COMMON LAYER OPERATIONS

--- Dense (Linear) Layers ---
    BERT-hidden fwd: 0.207 ± 0.819 ms (min=0.108, median=0.124)
    BERT-hidden fwd+bwd: 0.282 ± 1.007 ms (min=0.160, median=0.175)
    BERT-FFN-up fwd: 0.241 ± 0.749 ms (min=0.159, median=0.170)
    BERT-FFN-up fwd+bwd: 0.368 ± 1.006 ms (min=0.237, median=0.267)
    BERT-FFN-down fwd: 0.305 ± 0.752 ms (min=0.212, median=0.228)
    BERT-FFN-down fwd+bwd: 0.422 ± 1.004 ms (min=0.300, median=0.317)
    Large-FFN-up fwd: 0.167 ± 0.380 ms (min=0.127, median=0.140)
    Large-FFN-up fwd+bwd: 0.270 ± 0.204 ms (min=0.226, median=0.248)
    Large-FFN-down fwd: 0.425 ± 1.005 ms (min=0.297, median=0.323)
    Large-FFN-down fwd+bwd: 0.550 ± 1.005 ms (min=0.424, median=0.444)
    LLM-single fwd: 0.470 ± 1.010 ms (min=0.340, median=0.361)
    LLM-single fwd+bwd: 0.906 ± 1.414 ms (min=0.666, median=0.699)

--- BatchNorm ---
    BN-early: 0.392 ± 0.712 ms (min=0.321, median=0.338)
    BN-mid: 0.151 ± 0.342 ms (min=0.115, median=0.122)

## CNN Model Training

In [6]:
# ============================================================
# JAX BENCHMARK - Cell 6: CNN Model Training
# ============================================================
print("=" * 70)
print("BENCHMARK 5: CNN MODEL TRAINING (ResNet-like)")
print("=" * 70)

results['benchmarks']['cnn_training'] = {}

class ResidualBlock(nn.Module):
    filters: int
    strides: tuple = (1, 1)

    @nn.compact
    def __call__(self, x, train=True):
        residual = x

        y = nn.Conv(self.filters, (3, 3), strides=self.strides, padding='SAME', use_bias=False)(x)
        y = nn.BatchNorm(use_running_average=not train)(y)
        y = nn.relu(y)
        y = nn.Conv(self.filters, (3, 3), strides=(1, 1), padding='SAME', use_bias=False)(y)
        y = nn.BatchNorm(use_running_average=not train)(y)

        if self.strides != (1, 1) or residual.shape[-1] != self.filters:
            residual = nn.Conv(self.filters, (1, 1), strides=self.strides, use_bias=False)(residual)
            residual = nn.BatchNorm(use_running_average=not train)(residual)

        return nn.relu(y + residual)

class SimpleResNet(nn.Module):
    num_classes: int = 1000

    @nn.compact
    def __call__(self, x, train=True):
        x = nn.Conv(64, (7, 7), strides=(2, 2), padding='SAME', use_bias=False)(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.max_pool(x, (3, 3), strides=(2, 2), padding='SAME')

        # Layer 1
        x = ResidualBlock(64)(x, train=train)
        x = ResidualBlock(64)(x, train=train)
        # Layer 2
        x = ResidualBlock(128, strides=(2, 2))(x, train=train)
        x = ResidualBlock(128)(x, train=train)
        # Layer 3
        x = ResidualBlock(256, strides=(2, 2))(x, train=train)
        x = ResidualBlock(256)(x, train=train)
        # Layer 4
        x = ResidualBlock(512, strides=(2, 2))(x, train=train)
        x = ResidualBlock(512)(x, train=train)

        x = jnp.mean(x, axis=(1, 2))  # Global avg pool
        x = nn.Dense(self.num_classes)(x)
        return x

# Count parameters
key = random.PRNGKey(0)
dummy = jnp.ones([1, 224, 224, 3])
model = SimpleResNet(1000)
variables = model.init(key, dummy)
param_count = sum(x.size for x in jax.tree_util.tree_leaves(variables['params']))
print(f"\nModel: SimpleResNet-18-like")
print(f"Total parameters: {param_count:,}")
del dummy, variables

batch_sizes = [16, 32, 64]
num_classes = 1000

for batch_size in batch_sizes:
    label = f"batch_{batch_size}"
    print(f"\n--- Batch Size: {batch_size} ---")

    try:
        warmup_gpu()
        key, init_key, data_key = random.split(key, 3)

        model = SimpleResNet(num_classes)
        x = random.normal(data_key, (batch_size, 224, 224, 3))
        y = random.randint(random.PRNGKey(99), (batch_size,), 0, num_classes)

        variables = model.init(init_key, x)

        tx = optax.sgd(learning_rate=0.01, momentum=0.9)
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=variables['params'],
            tx=tx
        )
        batch_stats = variables.get('batch_stats', {})

        # Inference
        @jit
        def inference_fn(params, batch_stats, x):
            return model.apply({'params': params, 'batch_stats': batch_stats},
                             x, train=False, mutable=False)

        _ = inference_fn(state.params, batch_stats, x).block_until_ready()

        r_infer = benchmark_fn(lambda: inference_fn(state.params, batch_stats, x),
                              num_runs=50, warmup_runs=10, label=f"  Inference")
        throughput_infer = batch_size / (r_infer['mean_ms'] / 1000)
        r_infer['throughput_imgs_per_sec'] = float(throughput_infer)
        print(f"    → {throughput_infer:.1f} images/sec")

        # Training step
        @jit
        def train_step(state, batch_stats, x, y):
            def loss_fn(params):
                logits, updates = model.apply(
                    {'params': params, 'batch_stats': batch_stats},
                    x, train=True, mutable=['batch_stats']
                )
                loss = optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()
                return loss, updates

            (loss, updates), grads = value_and_grad(loss_fn, has_aux=True)(state.params)
            state = state.apply_gradients(grads=grads)
            batch_stats = updates['batch_stats']
            return state, batch_stats, loss

        state, batch_stats, _ = train_step(state, batch_stats, x, y)

        r_train = benchmark_fn(lambda: train_step(state, batch_stats, x, y),
                              num_runs=50, warmup_runs=10, label=f"  Training step")
        throughput_train = batch_size / (r_train['mean_ms'] / 1000)
        r_train['throughput_imgs_per_sec'] = float(throughput_train)
        print(f"    → {throughput_train:.1f} images/sec")

        results['benchmarks']['cnn_training'][label] = {
            'inference': r_infer,
            'training': r_train,
        }

        del model, state, x, y, variables, batch_stats
        gc.collect()

    except Exception as e:
        print(f"  SKIPPED: {e}")
        import traceback; traceback.print_exc()
        results['benchmarks']['cnn_training'][label] = {'error': str(e)}

# Mixed precision (FP16)
print(f"\n--- Mixed Precision Training (float16 compute) ---")
try:
    batch_size = 32
    warmup_gpu()
    key, init_key, data_key = random.split(key, 3)

    model = SimpleResNet(num_classes)
    x = random.normal(data_key, (batch_size, 224, 224, 3))
    y = random.randint(random.PRNGKey(99), (batch_size,), 0, num_classes)

    variables = model.init(init_key, x)
    tx = optax.sgd(learning_rate=0.01, momentum=0.9)
    state = train_state.TrainState.create(
        apply_fn=model.apply, params=variables['params'], tx=tx
    )
    batch_stats = variables.get('batch_stats', {})

    # Use float16 compute via jax.default_matmul_precision or manual cast
    @jit
    def amp_train_step(state, batch_stats, x, y):
        def loss_fn(params):
            x_half = x.astype(jnp.float16)
            logits, updates = model.apply(
                {'params': params, 'batch_stats': batch_stats},
                x_half, train=True, mutable=['batch_stats']
            )
            logits = logits.astype(jnp.float32)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()
            return loss, updates

        (loss, updates), grads = value_and_grad(loss_fn, has_aux=True)(state.params)
        state = state.apply_gradients(grads=grads)
        return state, updates.get('batch_stats', batch_stats), loss

    _, _, _ = amp_train_step(state, batch_stats, x, y)

    r_amp = benchmark_fn(lambda: amp_train_step(state, batch_stats, x, y),
                        num_runs=50, warmup_runs=10,
                        label=f"  AMP Training (batch={batch_size})")
    throughput_amp = batch_size / (r_amp['mean_ms'] / 1000)
    r_amp['throughput_imgs_per_sec'] = float(throughput_amp)
    print(f"    → {throughput_amp:.1f} images/sec")

    results['benchmarks']['cnn_training']['amp_batch_32'] = {
        'training': r_amp,
    }

    del model, state, x, y
    gc.collect()

except Exception as e:
    print(f"  AMP failed: {e}")
    import traceback; traceback.print_exc()

print("\n✅ CNN training benchmark complete!")

BENCHMARK 5: CNN MODEL TRAINING (ResNet-like)

Model: SimpleResNet-18-like
Total parameters: 11,689,512

--- Batch Size: 16 ---
    Inference: 18.265 ± 5.160 ms (min=13.348, median=15.689)
    → 876.0 images/sec
    Training step: 87.248 ± 5.118 ms (min=77.410, median=89.417)
    → 183.4 images/sec

--- Batch Size: 32 ---
    Inference: 27.069 ± 5.143 ms (min=19.725, median=29.639)
    → 1182.2 images/sec
    Training step: 150.772 ± 12.090 ms (min=134.845, median=149.396)
    → 212.2 images/sec

--- Batch Size: 64 ---
    Inference: 44.596 ± 4.238 ms (min=32.716, median=45.986)
    → 1435.1 images/sec
    Training step: 271.487 ± 5.997 ms (min=263.116, median=269.928)
    → 235.7 images/sec

--- Mixed Precision Training (float16 compute) ---
    AMP Training (batch=32): 154.760 ± 0.715 ms (min=152.716, median=154.715)
    → 206.8 images/sec

✅ CNN training benchmark complete!


## Transformer Model Training

In [7]:
# ============================================================
# JAX BENCHMARK - Cell 7: Transformer Model Training
# ============================================================
print("=" * 70)
print("BENCHMARK 6: TRANSFORMER MODEL TRAINING")
print("=" * 70)

results['benchmarks']['transformer_training'] = {}

class TransformerBlock(nn.Module):
    d_model: int
    nhead: int
    dim_ff: int

    @nn.compact
    def __call__(self, x, train=True):
        # Self-attention
        attn_out = nn.MultiHeadDotProductAttention(
            num_heads=self.nhead,
            qkv_features=self.d_model
        )(x, x)
        attn_out = nn.Dropout(rate=0.1, deterministic=not train)(attn_out)
        x = nn.LayerNorm()(x + attn_out)

        # FFN
        ffn = nn.Dense(self.dim_ff)(x)
        ffn = nn.relu(ffn)
        ffn = nn.Dense(self.d_model)(ffn)
        ffn = nn.Dropout(rate=0.1, deterministic=not train)(ffn)
        x = nn.LayerNorm()(x + ffn)

        return x

class TransformerClassifier(nn.Module):
    vocab_size: int = 30000
    d_model: int = 512
    nhead: int = 8
    num_layers: int = 6
    dim_ff: int = 2048
    max_seq_len: int = 512
    num_classes: int = 2

    @nn.compact
    def __call__(self, x, train=True):
        # Embedding
        x = nn.Embed(self.vocab_size, self.d_model)(x)
        x = x * jnp.sqrt(self.d_model).astype(x.dtype)

        # Positional encoding (learned)
        pos = self.param('pos_embedding',
                        nn.initializers.normal(0.02),
                        (1, self.max_seq_len, self.d_model))
        x = x + pos[:, :x.shape[1], :]

        # Transformer layers
        for _ in range(self.num_layers):
            x = TransformerBlock(
                d_model=self.d_model,
                nhead=self.nhead,
                dim_ff=self.dim_ff
            )(x, train=train)

        # Classification
        x = jnp.mean(x, axis=1)
        x = nn.Dense(self.num_classes)(x)
        return x

key = random.PRNGKey(999)

transformer_configs = [
    (256, 4, 4, 1024, 128, 64, "Small-Transformer"),
    (512, 8, 6, 2048, 128, 32, "BERT-base-like"),
    (512, 8, 6, 2048, 256, 16, "BERT-long-seq"),
    (768, 12, 6, 3072, 128, 16, "BERT-large-like"),
]

for d_model, nhead, num_layers, dim_ff, seq_len, batch_size, label in transformer_configs:
    print(f"\n--- {label} ---")
    print(f"  Config: d_model={d_model}, heads={nhead}, layers={num_layers}, "
          f"ff={dim_ff}, seq={seq_len}, batch={batch_size}")

    try:
        warmup_gpu()
        key, init_key, data_key, dropout_key = random.split(key, 4)

        model = TransformerClassifier(
            d_model=d_model, nhead=nhead, num_layers=num_layers,
            dim_ff=dim_ff, max_seq_len=seq_len
        )

        x = random.randint(data_key, (batch_size, seq_len), 0, 30000)
        y = random.randint(random.PRNGKey(42), (batch_size,), 0, 2)

        variables = model.init({'params': init_key, 'dropout': dropout_key}, x)

        param_count = sum(p.size for p in jax.tree_util.tree_leaves(variables['params']))
        print(f"  Parameters: {param_count:,}")

        tx = optax.adam(learning_rate=1e-4)
        state = train_state.TrainState.create(
            apply_fn=model.apply, params=variables['params'], tx=tx
        )

        # Inference
        @jit
        def infer_fn(params, x):
            return model.apply({'params': params}, x, train=False,
                             rngs={'dropout': random.PRNGKey(0)})

        _ = infer_fn(state.params, x).block_until_ready()

        r_infer = benchmark_fn(lambda: infer_fn(state.params, x),
                              num_runs=30, warmup_runs=10, label=f"  Inference")
        throughput_infer = batch_size / (r_infer['mean_ms'] / 1000)
        r_infer['throughput_samples_per_sec'] = float(throughput_infer)
        tokens_per_sec = (batch_size * seq_len) / (r_infer['mean_ms'] / 1000)
        r_infer['tokens_per_sec'] = float(tokens_per_sec)
        print(f"    → {throughput_infer:.1f} samples/sec, {tokens_per_sec:.0f} tokens/sec")

        # Training
        @jit
        def train_fn(state, x, y, dropout_key):
            def loss_fn(params):
                logits = model.apply({'params': params}, x, train=True,
                                    rngs={'dropout': dropout_key})
                loss = optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()
                return loss

            loss, grads = value_and_grad(loss_fn)(state.params)
            state = state.apply_gradients(grads=grads)
            return state, loss

        _, _ = train_fn(state, x, y, dropout_key)

        r_train = benchmark_fn(lambda: train_fn(state, x, y, dropout_key),
                              num_runs=30, warmup_runs=10, label=f"  Training step")
        throughput_train = batch_size / (r_train['mean_ms'] / 1000)
        r_train['throughput_samples_per_sec'] = float(throughput_train)
        tokens_per_sec_train = (batch_size * seq_len) / (r_train['mean_ms'] / 1000)
        r_train['tokens_per_sec'] = float(tokens_per_sec_train)
        print(f"    → {throughput_train:.1f} samples/sec, {tokens_per_sec_train:.0f} tokens/sec")

        results['benchmarks']['transformer_training'][label] = {
            'inference': r_infer,
            'training': r_train,
            'param_count': int(param_count),
            'config': {
                'd_model': d_model, 'nhead': nhead, 'layers': num_layers,
                'ff': dim_ff, 'seq_len': seq_len, 'batch': batch_size
            }
        }

        del model, state, x, y, variables
        gc.collect()

    except Exception as e:
        print(f"  SKIPPED: {e}")
        import traceback; traceback.print_exc()
        results['benchmarks']['transformer_training'][label] = {'error': str(e)}

print("\n✅ Transformer training benchmark complete!")

BENCHMARK 6: TRANSFORMER MODEL TRAINING

--- Small-Transformer ---
  Config: d_model=256, heads=4, layers=4, ff=1024, seq=128, batch=64
  Parameters: 10,872,322
    Inference: 14.702 ± 4.809 ms (min=11.020, median=11.794)
    → 4353.1 samples/sec, 557203 tokens/sec
    Training step: 59.671 ± 4.717 ms (min=55.597, median=57.764)
    → 1072.5 samples/sec, 137285 tokens/sec

--- BERT-base-like ---
  Config: d_model=512, heads=8, layers=6, ff=2048, seq=128, batch=32
  Parameters: 34,340,866
    Inference: 34.722 ± 5.484 ms (min=26.435, median=38.341)
    → 921.6 samples/sec, 117966 tokens/sec
    Training step: 120.549 ± 5.120 ms (min=114.209, median=118.121)
    → 265.5 samples/sec, 33978 tokens/sec

--- BERT-long-seq ---
  Config: d_model=512, heads=8, layers=6, ff=2048, seq=256, batch=16
  Parameters: 34,406,402
    Inference: 39.046 ± 4.614 ms (min=31.082, median=41.692)
    → 409.8 samples/sec, 104901 tokens/sec
    Training step: 132.882 ± 4.926 ms (min=124.939, median=134.135)
    

## JIT Compilation, Memory & Save

In [8]:
# ============================================================
# JAX BENCHMARK - Cell 8: JIT Compilation, Memory & Save
# ============================================================
print("=" * 70)
print("BENCHMARK 7: JIT COMPILATION TIME & MEMORY TRANSFER")
print("=" * 70)

results['benchmarks']['memory'] = {}
results['benchmarks']['jit_compile'] = {}

key = random.PRNGKey(0)

# --- JIT Compilation Time (JAX-specific) ---
print("\n--- JIT Compilation Time ---")
jit_configs = [
    ("matmul_1024", lambda k: (random.normal(k, (1024, 1024)),),
     lambda a: jnp.dot(a[0], a[0])),
    ("matmul_4096", lambda k: (random.normal(k, (4096, 4096)),),
     lambda a: jnp.dot(a[0], a[0])),
    ("conv_small", lambda k: (random.normal(k, (32, 32, 32, 3)),),
     lambda a: jax.lax.conv_general_dilated(a[0], random.normal(random.PRNGKey(0), (3, 3, 3, 64)), (1,1), 'SAME')),
]

for name, data_fn, compute_fn in jit_configs:
    try:
        key, subkey = random.split(key)
        data = data_fn(subkey)

        # Measure compilation time
        start = time.perf_counter()
        jitted_fn = jit(lambda: compute_fn(data))
        result = jitted_fn()
        if isinstance(result, jnp.ndarray):
            result.block_until_ready()
        compile_time = time.perf_counter() - start

        print(f"  {name}: {compile_time*1000:.1f} ms (first call with compilation)")

        # Measure post-compilation
        times = []
        for _ in range(50):
            start = time.perf_counter()
            r = jitted_fn()
            if isinstance(r, jnp.ndarray):
                r.block_until_ready()
            times.append(time.perf_counter() - start)

        post_compile = np.mean(times) * 1000
        print(f"  {name}: {post_compile:.3f} ms (post-compilation avg)")

        results['benchmarks']['jit_compile'][name] = {
            'first_call_ms': float(compile_time * 1000),
            'post_compile_ms': float(post_compile),
            'compilation_overhead_ms': float((compile_time * 1000) - post_compile)
        }

        del data
    except Exception as e:
        print(f"  {name}: ERROR - {e}")

# --- Memory Transfer ---
print("\n--- Memory Transfer (CPU ↔ GPU) ---")
transfer_sizes_mb = [1, 10, 50, 100, 500, 1000]

results['benchmarks']['memory']['h2d'] = {}
results['benchmarks']['memory']['d2h'] = {}

for size_mb in transfer_sizes_mb:
    numel = size_mb * 1024 * 1024 // 4

    try:
        # Host to Device
        cpu_array = np.random.randn(numel).astype(np.float32)

        def h2d_fn():
            gpu_array = jax.device_put(cpu_array)
            return gpu_array

        r_h2d = benchmark_fn(h2d_fn, num_runs=50, warmup_runs=10,
                            label=f"  H2D {size_mb} MB")
        bandwidth_h2d = size_mb / (r_h2d['mean_ms'] / 1000) / 1000
        r_h2d['bandwidth_gbs'] = float(bandwidth_h2d)
        print(f"    → {bandwidth_h2d:.2f} GB/s")
        results['benchmarks']['memory']['h2d'][f'{size_mb}MB'] = r_h2d

        # Device to Host
        gpu_array = jax.device_put(cpu_array)
        gpu_array.block_until_ready()

        def d2h_fn():
            return jax.device_get(gpu_array)

        r_d2h = benchmark_fn(d2h_fn, num_runs=50, warmup_runs=10,
                            label=f"  D2H {size_mb} MB")
        bandwidth_d2h = size_mb / (r_d2h['mean_ms'] / 1000) / 1000
        r_d2h['bandwidth_gbs'] = float(bandwidth_d2h)
        print(f"    → {bandwidth_d2h:.2f} GB/s")
        results['benchmarks']['memory']['d2h'][f'{size_mb}MB'] = r_d2h

        del cpu_array, gpu_array

    except Exception as e:
        print(f"  {size_mb} MB: ERROR - {e}")

# Allocation
print("\n--- GPU Memory Allocation ---")
results['benchmarks']['memory']['allocation'] = {}
for size_mb in [10, 100, 500, 1000]:
    numel = size_mb * 1024 * 1024 // 4
    try:
        def alloc_fn():
            t = jnp.zeros(numel, dtype=jnp.float32)
            t.block_until_ready()
            return t

        r = benchmark_fn(alloc_fn, num_runs=100, warmup_runs=20,
                        label=f"  Alloc {size_mb} MB")
        results['benchmarks']['memory']['allocation'][f'{size_mb}MB'] = r
    except:
        pass

# --- Save Results ---
print("\n" + "=" * 70)
print("SAVING RESULTS")
print("=" * 70)

results['timestamp'] = datetime.datetime.now().isoformat()

output_file = 'benchmark_jax_results.json'
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to: {output_file}")

# Summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

if 'matmul' in results['benchmarks']:
    if 'float32' in results['benchmarks']['matmul']:
        mm = results['benchmarks']['matmul']['float32']
        if '4096' in mm and 'tflops' in mm['4096']:
            print(f"MatMul 4096x4096 FP32: {mm['4096']['tflops']:.2f} TFLOPS")
    if 'float16' in results['benchmarks']['matmul']:
        mm = results['benchmarks']['matmul']['float16']
        if '4096' in mm and 'tflops' in mm['4096']:
            print(f"MatMul 4096x4096 FP16: {mm['4096']['tflops']:.2f} TFLOPS")

if 'cnn_training' in results['benchmarks']:
    if 'batch_32' in results['benchmarks']['cnn_training']:
        ct = results['benchmarks']['cnn_training']['batch_32']
        if 'training' in ct:
            print(f"ResNet Training (batch=32): {ct['training'].get('throughput_imgs_per_sec', 'N/A'):.1f} img/s")

if 'transformer_training' in results['benchmarks']:
    if 'BERT-base-like' in results['benchmarks']['transformer_training']:
        tt = results['benchmarks']['transformer_training']['BERT-base-like']
        if 'training' in tt:
            print(f"BERT Training: {tt['training'].get('tokens_per_sec', 'N/A'):.0f} tokens/sec")

print("\n🏁 JAX benchmark complete!")

BENCHMARK 7: JIT COMPILATION TIME & MEMORY TRANSFER

--- JIT Compilation Time ---
  matmul_1024: 200.7 ms (first call with compilation)
  matmul_1024: 0.067 ms (post-compilation avg)


2026-02-17 14:01:28.270578: E external/xla/xla/service/slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %dot_general.2 = f32[4096,4096]{1,0} dot(%constant.1, %constant.1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, metadata={op_name="jit(<lambda>)/dot_general" source_file="/tmp/ipykernel_149579/4254115883.py" source_line=19 source_end_line=19 source_column=15 source_end_column=34}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2026-02-17 14:01:29.344505: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 2.074032177s
Constant folding an instruction is taking > 1s:

  %dot_gene

  matmul_4096: 3726.8 ms (first call with compilation)
  matmul_4096: 0.687 ms (post-compilation avg)
  conv_small: ERROR - conv_general_dilated lhs feature dimension size divided by feature_group_count must equal the rhs input feature dimension size, but 32 // 1 != 3.

--- Memory Transfer (CPU ↔ GPU) ---
    H2D 1 MB: 0.232 ± 0.058 ms (min=0.198, median=0.228)
    → 4.31 GB/s
    D2H 1 MB: 0.005 ± 0.000 ms (min=0.004, median=0.004)
    → 220.82 GB/s
    H2D 10 MB: 3.615 ± 3.198 ms (min=1.738, median=2.266)
    → 2.77 GB/s
    D2H 10 MB: 0.005 ± 0.000 ms (min=0.004, median=0.005)
    → 2123.76 GB/s
    H2D 50 MB: 10.082 ± 2.967 ms (min=6.854, median=8.119)
    → 4.96 GB/s
    D2H 50 MB: 0.007 ± 0.002 ms (min=0.006, median=0.007)
    → 7122.73 GB/s
    H2D 100 MB: 17.299 ± 2.351 ms (min=13.586, median=17.599)
    → 5.78 GB/s
    D2H 100 MB: 0.005 ± 0.001 ms (min=0.004, median=0.004)
    → 21075.97 GB/s
    H2D 500 MB: 68.705 ± 3.115 ms (min=66.234, median=67.374)
    → 7.28 GB/s
    D2H